# RAG evaluation — FinVerify

This notebook runs our **implementation** financial advisor: **Claude (`claude-sonnet-4-6`)** augmented with retrieval over an authoritative knowledge base (IRS publications, CFPB guidance, SEC investor.gov, SSA, HealthCare.gov) persisted in a **ChromaDB** vector store.

Pipeline per question:

1. Embed the question with `BAAI/bge-small-en-v1.5`.
2. Retrieve top-k chunks from Chroma (`k=5`), filterable by topic when useful.
3. Build a prompt that includes retrieved passages with inline citation markers.
4. Call Claude with instructions to cite sources and refuse to fabricate when evidence is thin.
5. Grade with the same three signals as the baseline (MC accuracy, embedding similarity, LLM-as-judge) plus a **citation coverage** signal.

## 1. Prerequisites

```bash
python src/knowledge_base/ingest.py    # builds src/knowledge_base/vector_store/
export ANTHROPIC_API_KEY=sk-ant-...     # for Claude generation and the judge
```

In [ ]:
# %pip install -r ../../requirements.txt
import os, sys, json, time, re
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "dataset").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

# Load API keys from .env at repo root
try:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / ".env")
except ImportError:
    pass

from eval.sampling import sample_from_dataset
from eval.metrics import grade_mc, cosine_similarity, judge_with_claude

VECTOR_DIR = REPO_ROOT / "src" / "knowledge_base" / "vector_store"
assert VECTOR_DIR.exists(), "Vector store not found — run `python src/knowledge_base/ingest.py` first."
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in .env or your shell."
print("Repo root:", REPO_ROOT)

## 2. Sample the same 15 questions as the baseline

Using `seed=7` keeps the sample identical across notebooks so comparisons are apples-to-apples.

In [ ]:
N_PER_DATASET = 5
DATASETS = ["standard_questions", "open_ended_hard", "reddit_questions"]
sampled = []
for ds in DATASETS:
    sampled.extend(sample_from_dataset(ds, N_PER_DATASET, seed=7))
print(f"Total items: {len(sampled)}")

## 3. Connect to the vector store

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

chroma = chromadb.PersistentClient(path=str(VECTOR_DIR))
coll = chroma.get_collection("finverify_kb")
print(f"Collection size: {coll.count()} chunks")

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")


def retrieve(question: str, k: int = 5, topic: str | None = None) -> list[dict]:
    q_emb = embedder.encode([question], normalize_embeddings=True)[0].tolist()
    where = {"topic": topic} if topic else None
    res = coll.query(query_embeddings=[q_emb], n_results=k, where=where)
    hits = []
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        hits.append({"text": doc, "meta": meta, "distance": dist})
    return hits


# Sanity check
for h in retrieve("What is the difference between a traditional IRA and a Roth IRA?"):
    print(f"  [{h['meta']['publisher']}] {h['meta']['title']}  (d={h['distance']:.3f})")
    print(f"    {h['text'][:120]}")

## 4. RAG prompt

Each retrieved chunk is assigned an index `[1]`, `[2]`, .... Claude is instructed to cite chunk indices inline for every substantive claim and to say so explicitly when the passages don't contain enough evidence. This gives us a way to measure *citation coverage* later.

In [ ]:
SYSTEM_RAG = (
    "You are FinVerify, a careful personal-finance assistant. "
    "Answer using ONLY the provided passages when they are relevant. "
    "Cite sources inline as bracketed indices like [1], [2] after each claim they support. "
    "If the passages are insufficient to answer confidently, say so rather than fabricating. "
    "Acknowledge tradeoffs when they exist."
)

def build_rag_messages(item: dict, hits: list[dict]) -> list[dict]:
    ctx_lines = []
    for i, h in enumerate(hits, 1):
        m = h["meta"]
        ctx_lines.append(f"[{i}] ({m['publisher']} — {m['title']})\n{h['text']}")
    context_block = "\n\n".join(ctx_lines)

    if item.get("type") == "multiple_choice":
        options = "\n".join(item["options"])
        user = (
            f"Context:\n{context_block}\n\n"
            f"Question: {item['question']}\n\n{options}\n\n"
            "Respond with ONLY the single letter (A, B, C, or D)."
        )
    else:
        user = (
            f"Context:\n{context_block}\n\n"
            f"Question: {item['question']}\n\n"
            "Answer in 4-8 sentences. Cite passages as [n] after every substantive claim."
        )
    return user

## 5. Generate with Claude + RAG

In [ ]:
import anthropic

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-6"
K = 5

results = []
for i, item in enumerate(sampled, 1):
    t0 = time.time()
    hits = retrieve(item["question"], k=K, topic=item["_topic"])
    user_msg = build_rag_messages(item, hits)
    resp = client.messages.create(
        model=MODEL, max_tokens=700,
        system=SYSTEM_RAG,
        messages=[{"role": "user", "content": user_msg}],
    )
    answer = resp.content[0].text.strip()
    dt = time.time() - t0
    results.append({
        "id": item["id"], "dataset": item["_dataset"], "topic": item["_topic"],
        "type": item.get("type"), "difficulty": item.get("difficulty"),
        "question": item["question"],
        "reference": item["correct_answer"],
        "candidate": answer,
        "retrieved": [{"title": h["meta"]["title"], "publisher": h["meta"]["publisher"],
                       "source_slug": h["meta"]["source_slug"], "distance": h["distance"]}
                      for h in hits],
        "latency_sec": round(dt, 2),
    })
    print(f"{i:2d}/{len(sampled)}  {item['id']:<12}  {dt:5.1f}s  cites={len(re.findall(r'\[\d+\]', answer))}")

## 6. Score

### 6a. MC accuracy

In [ ]:
mc_rows = [r for r in results if r["type"] == "multiple_choice"]
for r in mc_rows:
    g = grade_mc(r["candidate"], r["reference"])
    r["mc_correct"] = g["is_correct"]; r["mc_picked"] = g["picked"]
if mc_rows:
    correct = sum(1 for r in mc_rows if r["mc_correct"])
    print(f"MC accuracy: {correct}/{len(mc_rows)} = {correct/len(mc_rows):.1%}")

### 6b. Embedding similarity (open-ended)

In [ ]:
oe_rows = [r for r in results if r["type"] != "multiple_choice"]
for r in oe_rows:
    a, b = embedder.encode([r["reference"], r["candidate"]], normalize_embeddings=True)
    r["embed_cosine"] = cosine_similarity(a, b)
if oe_rows:
    import statistics
    print(f"Mean cosine vs reference: {statistics.mean(r['embed_cosine'] for r in oe_rows):.3f}")

### 6c. LLM-as-judge rubric

In [ ]:
judge_client = anthropic.Anthropic()
for r in oe_rows:
    score = judge_with_claude(r["question"], r["reference"], r["candidate"], client=judge_client)
    r["judge"] = score.to_dict()
    print(f"{r['id']:<12}  f={score.factuality} c={score.completeness} a={score.advice_quality}  mean={score.mean:.2f}")

### 6d. Citation coverage

Counts how many of the retrieved passages the model actually cited. Low values mean the model is ignoring retrieval; high values mean it's using the grounded context.

In [ ]:
for r in results:
    cited = set(int(x) for x in re.findall(r"\[(\d+)\]", r["candidate"]))
    cited = {c for c in cited if 1 <= c <= len(r["retrieved"])}
    r["n_citations"] = len(cited)
    r["citation_coverage"] = len(cited) / max(len(r["retrieved"]), 1)
import statistics
print(f"Mean citation coverage: {statistics.mean(r['citation_coverage'] for r in results):.2f}")

## 7. Summary & side-by-side with the baseline

In [ ]:
import pandas as pd
df = pd.DataFrame(results)

def agg(g):
    row = {"n": len(g)}
    if "mc_correct" in g.columns:
        mc = g.dropna(subset=["mc_correct"])
        row["mc_accuracy"] = mc["mc_correct"].mean() if len(mc) else None
    if "embed_cosine" in g.columns:
        row["mean_cosine"] = g["embed_cosine"].mean()
    if "judge" in g.columns:
        judged = g.dropna(subset=["judge"])
        if len(judged):
            row["judge_mean"] = judged["judge"].apply(lambda j: j["mean"]).mean()
    if "citation_coverage" in g.columns:
        row["citation_coverage"] = g["citation_coverage"].mean()
    row["mean_latency_sec"] = g["latency_sec"].mean()
    return pd.Series(row)

rag_summary = df.groupby("dataset").apply(agg)
print("=== RAG ===")
print(rag_summary)

# Load baseline if present and align
baseline_path = next((REPO_ROOT / "src" / "notebooks" / "results").glob("baseline_*.json"), None)
if baseline_path:
    base = pd.DataFrame(json.loads(baseline_path.read_text()))
    print("\n=== Baseline ===")
    print(base.groupby("dataset").apply(agg))

## 8. Persist results

In [ ]:
OUT_DIR = REPO_ROOT / "src" / "notebooks" / "results"
OUT_DIR.mkdir(parents=True, exist_ok=True)
out_path = OUT_DIR / f"rag_{MODEL}.json"
out_path.write_text(json.dumps(results, indent=2))
print("Wrote", out_path)